# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/amanparganiha/flyrank-ml-internship/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [8]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
else:
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")

print("Working dir:", os.getcwd())
assert os.path.exists("data/raw/content_refresh_anonymized.csv"), "starter CSV not found"

import pandas as pd, numpy as np
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
print("Starter data found. Shape:", df.shape)

Working dir: /content/flyrank-ml-internship-starter/flyrank-ml-internship-starter
Starter data found. Shape: (30000, 44)


## 1. My lane (or freestyle) and why

*Name your lane — or say 'freestyle' and describe your own question. One short paragraph: why this one?*

Lane: CTR / Engagement Opportunity Scoring (predefined lane 4).

I chose it for three reasons.

It builds on something I already found. In notebook 01 I looked at CTR across
position tier and content type and hit an unexpected wall: median CTR is 0.0 in
almost every cell. The distribution is zero-inflated. That is a real analytical
problem rather than a lane I picked off a list, and it means the interesting work
starts immediately.

The action is cheap and reversible. A page that is visible but under-clicking can be
addressed by rewriting a title or meta description. No re-ranking, no new content, no
waiting on a crawl. Compare that with the refresh lane, where the recommended action
is "rewrite this article" — expensive, slow, and much costlier to get wrong.

It avoids the easiest trap in the dataset. The refresh lane's target is derived from
trend_direction, and notebook 02 showed exactly how that leaks. Building on the
starter pipeline's own label means inheriting its blind spots. A position-conditional
CTR gap is a label I define and can defend.

I am not choosing the refresh lane precisely because scripts/ already solves it end
to end. Reproducing the reference pipeline would teach me less.

In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
print("Rows:", len(df), "| Clients:", df["client_id"].nunique(),
      "| Content items:", df["content_id"].nunique())
print("\nCTR (%) distribution across all pages:")
print(df["ctr"].describe().round(3).to_string())
print("\nShare of pages with zero clicks in 90d:",
      round((df["clicks_90d"] == 0).mean(), 3))


Rows: 30000 | Clients: 32 | Content items: 30000

CTR (%) distribution across all pages:
count    30000.000
mean         0.511
std          3.279
min          0.000
25%          0.000
50%          0.070
75%          0.290
max        100.000

Share of pages with zero clicks in 90d: 0.44


## 2. The question: decision, action, cost of a wrong call

*What decision does your work improve? Who acts on it? What does a wrong recommendation cost?*

The question
Among pages that already have search visibility, which ones capture fewer clicks
than their position and content type would predict — and can a model rank those
pages better than a hand-written rule?

The decision it improves
A content team has limited review hours and thousands of pages. Someone has to decide
which pages get looked at this week. Today that decision is usually made by sorting
on traffic or on gut feel. My output changes what sits at the top of that list.

The action it supports
A ranked review queue with reason codes. A reviewer opens the top page and rewrites
its title and meta description. The page already has impressions, so the theory of
change is narrow and testable: convert existing visibility into clicks without
needing a ranking change.

The cost of a wrong call
Asymmetric, and worth being precise about.

A false positive costs roughly thirty minutes of a reviewer's time and a metadata
edit that may do nothing. Low cost, reversible.

A false negative — a genuinely under-capturing page that never surfaces — costs
whatever clicks it kept missing, indefinitely. Higher cost, but invisible, which
makes it the more dangerous error.

A third failure matters more than either: a queue that is confidently wrong. If the
model ranks by something spurious and a team reorganises their workflow around it,
the cost is trust rather than time. That is why the validation design is
client-holdout and why I compare against a hand rule on the same split rather than
reporting a standalone score.

Because false positives are cheap and false negatives are invisible, I optimise for
precision at the top of the queue rather than for overall accuracy — the top 50 is
all anyone will ever look at.

In [10]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
vis = df[df["impressions_90d"] >= 100]
print("Pages with >=100 impressions (addressable population):", len(vis))
print("Of those, share with zero clicks:", round((vis["clicks_90d"] == 0).mean(), 3))
print("\nIf a reviewer can do 50 pages/week, weeks to cover this population:",
      round(len(vis) / 50, 1))
print("-> the queue order is the whole product; nobody reaches the bottom.")

Pages with >=100 impressions (addressable population): 22006
Of those, share with zero clicks: 0.276

If a reviewer can do 50 pages/week, weeks to cover this population: 440.1
-> the queue order is the whole product; nobody reaches the bottom.


## 3. Quick look at the data (2-3 real numbers)

*Load the starter CSV below and show 2-3 real numbers that make your lane look worth the next 7 weeks.*

Three numbers that convinced me this lane is worth seven weeks.

1. The opportunity is large and concentrated. 22006 pages have 100+ impressions in the
   90-day window, and 27.6% of them get zero clicks. These are pages Google is already
   showing that nobody clicks — visibility that exists and is not converting.

2. Position does not explain it on its own. Median CTR by position tier runs 0.00%
   (deep), 0.16% (page_1), 0.03% (page_3_5), 0.11% (striking), 0.00% (top_3). Within
   page_1 alone, the spread across content types is far wider than the spread across
   tiers: feedly articles average 3.35% against comparison articles' 0.132%. Whatever
   separates them is not rank.

3. The population is workable. 15,615 pages have 100+ impressions in both the prev-30
   and last-30 windows, across 29 clients. Enough rows to model and enough clients for
   a grouped split — though 29 is few enough that a single split will be lumpy, which
   I note as a limitation now rather than discovering it later.

One thing I cannot yet explain: top_3 has a median CTR of 0.00%, identical to deep.
Either the tier does not mean what its name suggests, or zero-inflation dominates even
at the top of page one. I will resolve this in ML-04 when I write the data contract —
it directly affects whether position tier is usable as the basis for expected CTR.

In [11]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
print("Median CTR (%) by position tier:")
print(df.groupby("position_tier")["ctr"].median().round(3).to_string())

print("\nMean CTR (%) by content type, page_1 only:")
p1 = df[df["position_tier"] == "page_1"]
print(p1.groupby("content_type")["ctr"].mean().round(3).to_string())

vis = df[df["impressions_90d"] >= 100]
print("\nPages with 100+ impressions:", len(vis))
print("Share of those with zero clicks:", round((vis["clicks_90d"] == 0).mean(), 3))

both = df[(df["impressions_prev_30d"] >= 100) & (df["impressions_last_30d"] >= 100)]
print("\nPages with 100+ impressions in both 30d windows:", len(both))
print("Across clients:", both["client_id"].nunique())

print("\nOpen question — top_3 vs deep median CTR:")
print(df.groupby("position_tier")[["ctr","avg_position"]].median().round(3).to_string())


Median CTR (%) by position tier:
position_tier
deep        0.00
page_1      0.16
page_3_5    0.03
striking    0.11
top_3       0.00

Mean CTR (%) by content type, page_1 only:
content_type
comparison article    0.132
feedly article        3.350
keyword article       0.448

Pages with 100+ impressions: 22006
Share of those with zero clicks: 0.276

Pages with 100+ impressions in both 30d windows: 15615
Across clients: 29

Open question — top_3 vs deep median CTR:
                ctr  avg_position
position_tier                    
deep           0.00          61.0
page_1         0.16           6.6
page_3_5       0.03          28.9
striking       0.11          13.9
top_3          0.00           0.0


## 4. Careful words: what I can and can't claim

*Write what your work will be able to say (observed, directional, decision-support) — and what it never will (causal proof, 'predicting Google').*

What I can claim
That certain measurable attributes are associated with lower-than-expected CTR in
this anonymized snapshot. That a ranked queue built on those associations surfaces
under-capturing pages more efficiently than a hand rule, on clients the model has not
seen. That the ordering is useful for prioritising human review.

What I cannot claim
That any attribute causes low CTR. That rewriting a title will improve clicks — I
have no intervention data and no control group. Anything about how Google ranks or
scores pages; I observe outputs, not mechanisms. That results generalise beyond these
32 clients and this window.

Language I will use throughout: observed, measured, associated with, directional,
decision-support. Not: predicts, causes, proves, drives, Google's algorithm.

One specific caution I already know I need. In notebook 02 a depth-2 tree reached
Precision@50 = 1.000 by splitting on a feature the label was derived from. Reporting
that as a result would have been false. The same trap exists in this lane through any
last-30-window column, so my features are restricted to the prev-30 window and the
excluded list is written down rather than remembered.

In [12]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
print("Reminder — columns excluded from any model in this lane:")
for c in ["ctr","clicks_90d","clicks_last_30d","impressions_last_30d",
          "sessions_last_30d","trend_direction","trend_pct",
          "pageviews_90d","users_90d"]:
    print("  -", c)
print("\nReason: each is either in the outcome window or derived from the label.")

Reminder — columns excluded from any model in this lane:
  - ctr
  - clicks_90d
  - clicks_last_30d
  - impressions_last_30d
  - sessions_last_30d
  - trend_direction
  - trend_pct
  - pageviews_90d
  - users_90d

Reason: each is either in the outcome window or derived from the label.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.